In [0]:
from pyspark.sql import functions as F

In [0]:


def ingest_batch(source_table, target_path, batch_size):
    # First cell of your notebook logic
    spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")

    # 1. Get source columns and create the hash expression
    source_df_full = spark.read.table(source_table)
    all_columns = source_df_full.columns
    # cast("string") ensures we handle MAPs/STRUCTs without errors
    hash_expr = F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in all_columns]), 256)

    # 2. Physical check for Delta metadata to avoid [DELTA_TABLE_NOT_FOUND]
    delta_table_exists = False
    try:
        files = dbutils.fs.ls(target_path)
        if any(f.name == "_delta_log/" for f in files):
            delta_table_exists = True
    except:
        delta_table_exists = False

    # 3. Get already loaded record hashes
    if delta_table_exists:
        existing_ids = spark.read.format("delta").load(target_path) \
                            .select(hash_expr.alias("row_hash"))
        print("Checking for next available batch...")
    else:
        existing_ids = spark.createDataFrame([], "row_hash STRING")
        print("Target empty. Starting initial load...")

    # 4. Filter for new data, apply the batch_size limit
    new_data = source_df_full.withColumn("row_hash", hash_expr) \
        .join(existing_ids, "row_hash", "left_anti") \
        .limit(batch_size) \
        .drop("row_hash") \
        .withColumn(
        "load_timestamp", 
        F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss.SSS"))

    # 5. Write and Finish
    batch_count = new_data.count()
    if batch_count > 0:
        print(f"Loading {batch_count} rows...")
        new_data.write.format("delta").mode("append").save(target_path)
        print("Batch complete. Notebook finishing.")
    else:
        print("No new data found. Target is fully synchronized.")

    # Notebook ends here. Next run will pick up the next batch.